# Agent Nodes Prototype
Build and validate the 3 LLM nodes + graph wiring here before migrating to `src/agents/nodes/`.

**Order:**
1. Setup & state contract reminder
2. `rule_extraction` node
3. Validate `raw_rules` output
4. `rule_structuring` node
5. Validate `structured_rules` / `low_confidence_rules` split
6. `human_review` node (mock interrupt)
7. `graph.py` wiring
8. End-to-end smoke test

## 1. Setup

In [1]:
from dotenv import load_dotenv
_ = load_dotenv()

In [2]:
import sys
import os
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage # noqa: F401
from langchain.tools import tool # noqa: F401
from langchain_groq import ChatGroq
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate, SystemMessagePromptTemplate # noqa: F401
from langgraph.graph import StateGraph, END # noqa: F401
from langgraph.checkpoint.memory import InMemorySaver # noqa: F401
from src.docs_processing.docs_processor import DocumentProcessor
from src.models.compilance_rules import ComplianceRuleModel, RuleExtractionOutput


# Make sure the workspace root is on the path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('Root:', ROOT)

c:\Users\777kr\Desktop\Data Compliance Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'src'

In [ ]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,
    check_every_n_seconds=0.1,
    max_bucket_size=10)

In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7,
    timeout=20,
    rate_limiter=rate_limiter).bind(logprobs=True)

In [ ]:
callback = UsageMetadataCallbackHandler()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import logging

# ── Suppress noisy src.* loggers — show only warnings and above ──────────────
logging.getLogger("src").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("groq").setLevel(logging.WARNING)

# ---------------------------------------------------------------------------
# System Prompt
# ---------------------------------------------------------------------------
RULE_EXTRACTION_SYSTEM_PROMPT = """\
You are an expert **Data Compliance Officer** and **Database Architect**.
Your goal is to analyze policy documents and extract actionable, deterministic rules that can be validated against database records.

### INSTRUCTIONS
1.  **Analyze the Text**: Read the provided text chunk efficiently.
2.  **Filter for Mandates**: Ignore informational text, definitions, or broad principles. Focus ONLY on actionable constraints (look for "must", "shall", "required", "prohibited", "retain for", "encrypt").
3.  **Extract Rules**: For each mandate, populate a structured rule object.

### EXTRACTION LOGIC
- **Rule Text**: Quote the specific sentence(s) from the text.
- **Rule Type**: Classify as one of EXACTLY these values (no other values are valid):
    - `data_retention`  — how long data is kept (e.g., "retain for 7 years").
    - `data_quality`    — format or validity (e.g., "email must contain @").
    - `data_privacy`    — access control or data minimization.
    - `data_security`   — encryption, hashing, transmission controls.
    - `data_access`     — who is permitted to read / write data.
- **Logic Extraction** (CRITICAL):
    - Formulate a machine-testable logic condition wherever possible.
    - If the rule says "Users must be over 18", the logic is `field="age"`, `operator=">="`, `value="18"`.
    - If the rule is abstract (e.g., "Data must be secure"), still capture the rule text but leave logic fields empty strings.

### HANDLING AMBIGUITY
- If a rule implies a timeframe but doesn't specify it, capture the rule text and leave `timeframe` empty.
- If a specific data element is mentioned (e.g., "SSN", "Credit Card"), capture it in `scope`.

### OUTPUT FORMAT
You must output a structured object compatible with `RuleExtractionOutput`.
`rule_type` MUST be one of: data_retention, data_quality, data_privacy, data_security, data_access.
"""

rule_extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", RULE_EXTRACTION_SYSTEM_PROMPT),
    ("human", "Analyze the following policy text and extract compliance rules:\n\n{chunk_content}"),
])

print("Prompt ready. Rule types enforced: data_retention | data_quality | data_privacy | data_security | data_access")


Prompt ready. Rule types enforced: data_retention | data_quality | data_privacy | data_security | data_access


## 2. Rule Extraction Node

**Goal:** Read the policy PDF, chunk it, and use an LLM to extract `ComplianceRuleModel` objects.

**Pattern:**
```python
llm.with_structured_output(RuleExtractionOutput)
```

The existing `DocumentProcessor` already extracts and chunks the PDF — reuse it.

**State output your node must produce:**
```python
{"raw_rules": List[ComplianceRuleModel]}
```

In [ ]:
def rule_extraction_node(state):
    processor = DocumentProcessor()
    chunks = processor.process_pdf(state["document_path"])

    # Strip logprobs binding — incompatible with with_structured_output
    base_model = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0,          # deterministic extraction
        timeout=20,
        rate_limiter=rate_limiter,
    )
    structured_llm = base_model.with_structured_output(RuleExtractionOutput)

    # Chain: prompt template → structured LLM
    chain = rule_extraction_prompt | structured_llm

    all_rules = []
    seen_ids: set[str] = set()

    for i, chunk in enumerate(chunks):
        try:
            result: RuleExtractionOutput = chain.invoke({"chunk_content": chunk.content})
            for rule in result.extracted_rules:
                if rule.rule_id not in seen_ids:          # deduplicate
                    seen_ids.add(rule.rule_id)
                    all_rules.append(rule)
        except Exception as e:
            print(f"  [WARN] chunk {i} failed: {e!r} — skipping")

    return {"raw_rules": all_rules, "current_stage": "rules_extracted"}


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[..., list[AnyMessage]]

### Validate `raw_rules` output
Run this cell after implementing `rule_extraction_node` above. It checks the output matches the state contract.

In [ ]:
# ⚠️  Replace "your_policy.pdf" with an actual compliance policy document.
#     raft.pdf is a distributed systems paper — the LLM will extract 0 rules from it.
POLICY_PDF = os.path.join(ROOT, "data", "raft.pdf")   # <-- change this

if not os.path.exists(POLICY_PDF):
    raise FileNotFoundError(
        f"Policy PDF not found: {POLICY_PDF}\n"
        "Put a compliance policy PDF in data/ and update POLICY_PDF above."
    )

test_state = {"document_path": POLICY_PDF}
extraction_result = rule_extraction_node(test_state)

assert extraction_result is not None, "Node returned None"
assert "raw_rules" in extraction_result, "Missing 'raw_rules' key"
assert isinstance(extraction_result["raw_rules"], list), "raw_rules must be a list"
assert len(extraction_result["raw_rules"]) > 0, (
    "No rules extracted — check that the PDF contains compliance mandates "
    "(phrases like 'must', 'shall', 'prohibited')"
)

for i, r in enumerate(extraction_result["raw_rules"]):
    assert isinstance(r, ComplianceRuleModel), f"Item {i} is not a ComplianceRuleModel"

print(f"OK — extracted {len(extraction_result['raw_rules'])} rules")
for r in extraction_result["raw_rules"][:3]:
    print(f"  [{r.rule_type}] {r.rule_id}: {r.rule_text[:80]}")


[02/21/26 18:23:06] INFO     DocumentProcessor: Cache DISABLED                                 docs_processor.py:74

                    INFO     Processing PDF: raft.pdf                                         docs_processor.py:128

                    INFO     Total pages in document: 11                                      docs_processor.py:153

                    INFO     Extracted text from page 1                                       docs_processor.py:158

                    INFO     Created chunk raft.pdf_0_2c889c2c from raft.pdf page 1 (chars    docs_processor.py:241
                             0-964)                                                                                

...

                    INFO     Reached end of text for raft.pdf page 11                         docs_processor.py:246

                    INFO     Finished processing PDF: 53 chunks from 11 pages                 docs_processor.py:183

                    INFO     Chunk details: [{'source': 'raft.pdf', 'chunk_id':               docs_processor.py:189
                             'raft.pdf_0_2c889c2c', 'chunk_index': 0, 'page_number': 1,                            
                             'char_range': '0-964', 'total_pages': 11, 'page_width': 612.0,                        
                             'page_height': 792.0, 'extracted_at':                                                 
                             '2026-02-21T18:23:06.787755', 'section_header': None,                                 
                             'has_obligation_language': False}, {'source': 'raft.pdf',                             
                             'chunk_id': 'raft.pdf_1_62caaf00', 'chunk_index': 1,                                  
                             'page_number': 1, 'char_range': '965-1942', 'total_pages': 11,                        
                             'page_width': 612.0, 'page_height': 792.0, 'extracted_at':                            
                             '2026-02-21T18:23:06.787755', 'section_header': None,                                 
                             'has_obligation_language': False}, {'source': 'raft.pdf',                             
                             'chunk_id': 'raft.pdf_2_90a18e35', 'chunk_index': 2,                                  
                             'page_number': 1, 'char_range': '1943-2927', 'total_pages': 11,                       
                             'page_width': 612.0, 'page_height': 792.0, 'extracted_at':                            
                             '2026-02-21T18:23:06.787755', 'section_header': None,                                 
                             'has_obligation_language': False}, {'source': 'raft.pdf',                             
                             'chunk_id': 'raft.pdf_3_81d013b6', 'chunk_index': 3,                                  
                             'page_number': 1, 'char_range': '2928-3891', 'total_pages': 11,                       
                             'page_width': 612.0, 'page_height': 792.0, 'extracted_at':                            
                             '2026-02-21T18:23:06.787755', 'section_header': None,                                 
                             'has_obligation_language': False}, {'source': 'raft.pdf',                             
                             'chunk_id': 'raft.pdf_4_81f759e2', 'chunk_index': 4,                                  
                             'page_number': 1, 'char_range': '3892-4258', 'total_pages': 11,                       
                             'page_width': 612.0, 'page_height': 792.0, 'extracted_at':                            
                             '2026-02-21T18:23:06.787755', 'section_header': None,                                 
                             'has_obligation_language': False}, {'source': 'raft.pdf',                             
                             'chunk_id': 'raft.pdf_0_d95add3b', 'chunk_index': 0,                                  
                             'page_number': 2, 'char_range': '0-967', 'total_pages': 11,                           
                             'page_width': 612.0, 'page_height': 792.0, 'extracted_at':                            
                             '2026-02-21T18:23:06.814819', 'section_header': None,                                 
                             'has_obligation_language': False}, {'source': 'raft.pdf',                             
                             'chunk_id': 'raft.pdf_1_3b2dc2ac', 'chunk_index': 1,                                  
                             'page_number': 2, 'char_range': '968-1955', 'total_pages': 11,                        
                             'page_width': 612.0, 'page_height': 792.0, 'extracted_at':                            
                             '2026-02-21T18:23:06.814819

  [WARN] chunk 0 failed: BadRequestError('Error code: 400 - {\'error\': {\'message\': \'tool call validation failed: parameters for tool RuleExtractionOutput did not match schema: errors: [`/extracted_rules/0/timeframe_days`: expected integer, but got string, `/extracted_rules/0/timeframe_days`: expected null, but got string, `/extracted_rules/1/timeframe_days`: expected integer, but got string, `/extracted_rules/1/timeframe_days`: expected null, but got string]\', \'type\': \'invalid_request_error\', \'code\': \'tool_use_failed\', \'failed_generation\': \'<function=RuleExtractionOutput> {"document_type": "requirement", "entities": {}, "extracted_rules": [{"rule_id": "1", "rule_text": "the model to ignore those documents that don’t", "rule_type": "data_access", "confidence": 0.8, "logic": {"field": "", "operator": "", "value": ""}, "penalty": "", "scope": "", "source_reference": "", "timeframe": "", "timeframe_days": ""}, {"rule_id": "2", "rule_text": "answer questions in an open-book 

KeyboardInterrupt: 

## 3. Rule Structuring Node

**Goal:** For each `ComplianceRuleModel`, ask the LLM to map it to a real database column.

**Pattern:**
```python
llm.with_structured_output(StructuredRule)
```

The prompt must include:
- `rule_text` (the natural language rule)
- All available column names from `schema_metadata` (so LLM can pick a real one)

**State output your node must produce:**
```python
{
    "structured_rules":      List[StructuredRule],  # confidence >= 0.7
    "low_confidence_rules":  List[StructuredRule],  # confidence <  0.7
}
```

In [ ]:
from src.models.structured_rule import StructuredRule

CONFIDENCE_THRESHOLD = 0.7

# ── YOUR CODE BELOW ──────────────────────────────────────────────────────────
# Suggested structure:
#
# def rule_structuring_node(state):
#     raw_rules     = state["raw_rules"]
#     schema        = state["schema_metadata"]
#
#     # Flatten all column names across all tables for the prompt
#     all_columns = [
#         {"table": t, "column": c["column_name"], "type": c["data_type"]}
#         for t, info in schema.items()
#         for c in info["columns"]
#     ]
#
#     llm = ...                            # ChatAnthropic / ChatOpenAI
#     structured_llm = llm.with_structured_output(StructuredRule)
#
#     structured, low_confidence = [], []
#     for rule in raw_rules:
#         result = structured_llm.invoke(<prompt with rule.rule_text + all_columns>)
#         if result.confidence >= CONFIDENCE_THRESHOLD:
#             structured.append(result)
#         else:
#             low_confidence.append(result)
#
#     return {
#         "structured_rules":     structured,
#         "low_confidence_rules": low_confidence,
#         "current_stage":        "rules_structured",
#     }

def rule_structuring_node(state):
    pass  # replace with your implementation

### Validate `rule_structuring` output

In [ ]:
# Pull schema from a real DB to test against real column names
from src.agents.tools.database.sqlite_connector import SQLiteConnector

db_path = os.path.join(ROOT, "data", "HI-Small_Trans.db")
conn = SQLiteConnector(db_path)
conn.connect()
schema = conn.discover_schema()
conn.close()

print("Tables:", list(schema.keys()))
for t, info in schema.items():
    cols = [c["column_name"] for c in info["columns"]]
    print(f"  {t}: {cols}")

In [ ]:
# Run rule_structuring_node with real schema + extracted rules from cell-04
structuring_input = {
    "raw_rules": extraction_result["raw_rules"],
    "schema_metadata": schema,
}

structuring_result = rule_structuring_node(structuring_input)

assert structuring_result is not None, "Node returned None"
assert "structured_rules" in structuring_result,     "Missing 'structured_rules'"
assert "low_confidence_rules" in structuring_result, "Missing 'low_confidence_rules'"

for r in structuring_result["structured_rules"]:
    assert isinstance(r, StructuredRule), f"Not a StructuredRule: {r}"
    assert r.target_column, f"Rule '{r.rule_id}' has empty target_column"
    assert r.confidence >= CONFIDENCE_THRESHOLD, f"High-conf list has rule with conf {r.confidence}"

for r in structuring_result["low_confidence_rules"]:
    assert r.confidence < CONFIDENCE_THRESHOLD

print(f"OK — {len(structuring_result['structured_rules'])} ready, "
      f"{len(structuring_result['low_confidence_rules'])} need review")

for r in structuring_result["structured_rules"]:
    print(f"  [{r.confidence:.2f}] {r.rule_id}: {r.target_column} {r.operator} {r.value}")

## 4. Human Review Node

**Goal:** Pause the graph for low-confidence rules, let a human approve / edit / drop them, then merge back into `structured_rules`.

In the real graph this uses `interrupt()` from LangGraph. In the notebook we simulate by passing a `mock_decision` dict.

**State output your node must produce:**
```python
{
    "structured_rules": original_structured + approved/edited,  # merged
    "review_decision":  {"approved": [...], "edited": [...], "dropped": [...]},
}
```

In [ ]:
from langgraph.types import interrupt

# ── YOUR CODE BELOW ──────────────────────────────────────────────────────────
# Suggested structure:
#
# def human_review_node(state):
#     low = state.get("low_confidence_rules", [])
#     if not low:
#         return {}   # nothing to review, pass through
#
#     # Pause graph — surfaces to caller as result["__interrupt__"]
#     decision = interrupt({
#         "message": "The following rules have low confidence. Review each.",
#         "rules": [r.__dict__ for r in low],
#     })
#     # decision = {"approved": [rule_id, ...], "edited": [{rule_id, changes}], "dropped": [rule_id, ...]}
#
#     approved_ids = set(decision.get("approved", []))
#     dropped_ids  = set(decision.get("dropped",  []))
#     edited_map   = {e["rule_id"]: e for e in decision.get("edited", [])}
#
#     new_structured = []
#     for r in low:
#         if r.rule_id in dropped_ids:
#             continue
#         if r.rule_id in edited_map:
#             # apply edits — update fields from the decision dict
#             edits = edited_map[r.rule_id]
#             for field, val in edits.items():
#                 if field != "rule_id" and hasattr(r, field):
#                     setattr(r, field, val)
#         r.confidence = 1.0   # human approved = full confidence
#         new_structured.append(r)
#
#     return {
#         "structured_rules": state.get("structured_rules", []) + new_structured,
#         "review_decision":  decision,
#         "current_stage":    "review_complete",
#     }

def human_review_node(state):
    pass  # replace with your implementation


# ── NOTEBOOK MOCK (no real interrupt here) ──────────────────────────────────
def mock_human_review(state, mock_decision):
    """Simulate interrupt resume for notebook testing."""
    low = state.get("low_confidence_rules", [])
    if not low:
        print("No low-confidence rules to review.")
        return state

    print(f"Rules needing review ({len(low)}):")
    for r in low:
        print(f"  [{r.confidence:.2f}] {r.rule_id}: {r.target_column} {r.operator} {r.value}")

    print("\nMock decision:", mock_decision)
    # Apply mock decision same way the real node will
    approved_ids = set(mock_decision.get("approved", []))
    dropped_ids  = set(mock_decision.get("dropped",  []))
    edited_map   = {e["rule_id"]: e for e in mock_decision.get("edited", [])}

    new_structured = []
    for r in low:
        if r.rule_id in dropped_ids:
            print(f"  DROPPED: {r.rule_id}")
            continue
        if r.rule_id in edited_map:
            edits = edited_map[r.rule_id]
            for field, val in edits.items():
                if field != "rule_id" and hasattr(r, field):
                    setattr(r, field, val)
            print(f"  EDITED:  {r.rule_id}")
        elif r.rule_id in approved_ids:
            print(f"  APPROVED:{r.rule_id}")
        r.confidence = 1.0
        new_structured.append(r)

    return {
        "structured_rules": state.get("structured_rules", []) + new_structured,
        "review_decision":  mock_decision,
    }

In [ ]:
# Test mock review
low_conf = structuring_result.get("low_confidence_rules", [])

if low_conf:
    # Approve first rule, drop the rest for this test
    mock_decision = {
        "approved": [low_conf[0].rule_id] if low_conf else [],
        "edited":   [],
        "dropped":  [r.rule_id for r in low_conf[1:]],
    }
    reviewed = mock_human_review(structuring_result, mock_decision)
    print(f"\nFinal structured_rules count: {len(reviewed['structured_rules'])}")
else:
    print("All rules had sufficient confidence — human_review node would be skipped")
    reviewed = structuring_result

## 5. Graph Wiring

Wire all 6 nodes into the `StateGraph`. The conditional edge after `rule_structuring` is the key routing logic.

```
START → rule_extraction → schema_discovery → rule_structuring
                                                    │
                            low_confidence_rules? ──┤
                                   YES              ▼
                                             human_review
                                   NO              │
                                    └──────────────┘
                                                    ▼
                                           data_scanning
                                                    ▼
                                        violation_reporting
                                                    ▼
                                                   END
```

In [ ]:
"""
Graph wiring — now using the production modules from src/agents/.
No more stubs or temporary bypass edges.
"""
from src.agents.graph import build_graph
from src.agents.memory import get_checkpointer
from src.agents.runtime import make_config
from src.agents.streaming import UsageTracker, stream_graph_updates

# ── Build graph with in-memory checkpointer ─────────────────────────────────
with get_checkpointer("memory") as cp:
    graph = build_graph(checkpointer=cp)

    print("Graph compiled OK")
    print("Nodes:", list(graph.nodes))

    # Quick visual of the graph structure
    try:
        graph.get_graph().print_ascii()
    except Exception:
        pass  # print_ascii may not be available in all versions

## 6. End-to-End Smoke Test
Run the partial graph (schema_discovery → data_scanning → violation_reporting) with manually supplied rules to confirm the deterministic nodes work end-to-end. Swap in your nodes as you complete them.

In [ ]:
"""
End-to-end smoke test — runs the FULL pipeline:
  rule_extraction → schema_discovery → rule_structuring → data_scanning → violation_reporting

Uses production modules: memory, streaming, runtime.
"""
from src.agents.graph import build_graph
from src.agents.memory import get_checkpointer
from src.agents.runtime import make_config
from src.agents.streaming import UsageTracker, stream_graph_updates
from src.agents.nodes.violation_reporting import print_report

# ⚠️  CHANGE THESE to real paths in your environment
POLICY_PDF = os.path.join(ROOT, "data", "your_policy.pdf")  # <-- real compliance PDF
DB_PATH    = os.path.join(ROOT, "data", "HI-Small_Trans.db")

initial_state = {
    "document_path": POLICY_PDF,
    "db_type":       "sqlite",
    "db_config":     {"db_path": DB_PATH},
    "violations_db_path": os.path.join(ROOT, "violations.db"),
    "batch_size":    500,
    "errors":        [],
}

# ── Run with streaming + usage tracking ──────────────────────────────────────
tracker = UsageTracker()
config = make_config(
    thread_id="smoke-test-001",
    callbacks=[tracker],
    tags=["notebook", "smoke-test"],
)

with get_checkpointer("memory") as cp:
    graph = build_graph(checkpointer=cp)
    final_state = stream_graph_updates(graph, initial_state, config)

# ── Print results ────────────────────────────────────────────────────────────
print("\n--- Usage ---")
print(tracker.summary())

print("\nStage:", final_state.get("current_stage"))
print("Errors:", final_state.get("errors", []))

report = final_state.get("violation_report", {})
if report and not report.get("error"):
    print_report(report)
else:
    print("Report:", report)

## 7. Migration Checklist

Once a node works correctly in this notebook, copy it to the right file:

| Node | Notebook cell | Destination file |
|---|---|---|
| `rule_extraction_node` | cell 3 | `src/agents/nodes/rule_extraction.py` |
| `rule_structuring_node` | cell 5 | `src/agents/nodes/rule_structuring.py` |
| `human_review_node` | cell 8 | `src/agents/nodes/human_review.py` |
| `build_graph()` | cell 10 | `src/agents/graph.py` |

After migrating each node:
1. Add it to `src/agents/nodes/__init__.py`
2. Uncomment the corresponding lines in `build_graph()` (cell 10)
3. Remove the temporary bypass edges